In [1]:
import torch
import tqdm
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from sklearn.metrics import precision_recall_fscore_support
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
!nvidia-smi

Thu Nov 21 09:32:01 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3070 Ti     Off | 00000000:06:00.0 Off |                  N/A |
|  0%   35C    P8              20W / 290W |   7065MiB /  8192MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
class ResNet6(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet6, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=4, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.max_pool2d(x, 2)
        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [4]:
# Check that MPS is available
device = torch.device("cpu")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("DEVICE = MPS")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("DEVICE = CUDA")

DEVICE = CUDA


In [5]:
def set_seed(seed):
    # Set seed for Python's built-in random number generator
    torch.manual_seed(seed)
    # Set seed for numpy
    np.random.seed(seed)
    # Set seed for CUDA if available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Set cuDNN's random number generator seed for deterministic behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Usage example:
set_seed(69)

In [6]:
# EarlyStopping class to stop training when validation performance plateaus
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path='best_model.pth'):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 5
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path to save the best model. Default: 'best_model.pth'
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decreases.'''
        if self.verbose:
            print(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [7]:
import torch
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split


def create_optimizer(params, model):
    """
    Creates and returns the optimizer based on the parameter configuration.
    Args:
        params (dict): Contains 'optimizer', 'lr', and other hyperparameters.
        model (nn.Module): The model to optimize.
    Returns:
        optimizer (torch.optim.Optimizer): The initialized optimizer.
    """
    if params['optimizer'] == 'Adam':
        return optim.Adam(model.parameters(), lr=params['lr'])
    if params['optimizer'] == 'AdamW':
        return optim.AdamW(model.parameters(), lr=params['lr'])
    elif params['optimizer'] == 'SGD':
        return optim.SGD(model.parameters(), lr=params['lr'], momentum=0.9)
    elif params['optimizer'] == 'RMSprop':
        return optim.RMSprop(model.parameters(), lr=params['lr'], momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {params['optimizer']}")

# Custom training function with early stopping, model saving, and optimizer selection
def train_and_validate(model, train_loader, val_loader, config, early_stopping):
    best_loss = np.inf
    best_f1 = 0
    best_model_path = early_stopping.path
    
    # Create optimizer based on the config
    optimizer = create_optimizer(config, model)
    
    # Scheduler can be used for learning rate adjustments
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.9)

    for epoch in range(config['num_epochs']):
        model.train()
        train_loss = 0.0
        
        for batch_idx, (data, targets, _) in enumerate(train_loader):
            data, targets = data.to(config['device']), targets.to(config['device'])

            optimizer.zero_grad()
            outputs = model(data)
            pos_weight = targets * config['positive_weight_factor']
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        print(f"Epoch {epoch+1}/{config['num_epochs']}, Train Loss: {train_loss}")
        wandb.log({"train/loss": train_loss, "train/lr": optimizer.param_groups[0]['lr']})

        # Validation
        val_loss, val_f1, val_precision, val_recall = validate(model, val_loader, config)
        wandb.log({"val/loss": val_loss, "val/sF1@25": val_f1, "val/sPrecision@25": val_precision, "val/sRecall@25": val_recall})

        # Early stopping checks
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

        scheduler.step(val_loss)
        print("Scheduler:", scheduler.state_dict())
        
# Updated test loop to calculate AUC
def test_model(model, test_loader, config, seed):
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    # Load the best model saved by early stopping
    model.load_state_dict(torch.load(f"./models/{config['architecture']}-{config['predictor']}-bestF1-seed_{seed}_opt_{config['optimizer']}_lr_{config['lr']}_batch_{config['batch_size']}_weight{config['positive_weight_factor']}.pth"))

    with torch.no_grad():
        for data, targets, _ in test_loader:
            data = data.to(config["device"])
            targets = targets.to(config["device"])

            outputs = model(data)
            pos_weight = targets * config["positive_weight_factor"]
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            test_loss += criterion(outputs, targets).item()

            # Collect raw predictions (probabilities after sigmoid) for AUC calculation
            probabilities = torch.sigmoid(outputs).cpu().numpy()
            all_predictions.append(probabilities)
            all_targets.append(targets.cpu().numpy())

    # Average the test loss
    test_loss /= len(test_loader)

    # Concatenate all predictions and targets
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate AUC (average='macro' or 'micro' may vary depending on your needs)
    # try:
        
    zero_cols_targets = np.all(all_targets == 0, axis=0)
    ones_cols_targets = np.all(all_targets == 1, axis=0)
    zero_cols = zero_cols_targets | ones_cols_targets

    filtered_targets = all_targets[:][:, ~zero_cols]

    list_of_lists = np.asarray([arr.tolist() for arr in all_predictions])
    filtered_probas = list_of_lists[:][:, ~zero_cols]

    # Calculate and print AUC (Area Under ROC Curve)
    auc = roc_auc_score(filtered_targets, filtered_probas, average='macro')

    # except ValueError:
    #     auc = None  # AUC calculation requires at least one positive and one negative sample

    # Calculate Precision, Recall, F1 scores using thresholding on predictions
    top_k_indices = np.argsort(-all_predictions, axis=1)[:, :25]  # Top-25 predictions per sample
    thresholded_predictions = np.zeros_like(all_predictions)
    rows = np.arange(len(all_predictions))[:, None]
    thresholded_predictions[rows, top_k_indices] = 1

    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, thresholded_predictions, average='samples')

    # Log the results to W&B
    wandb.log({
        "test/loss": test_loss,
        "test/sF1@25": f1,
        "test/sPrecision@25": precision,
        "test/sRecall@25": recall,
        "test/AUC": auc
    })

    print(f"Test Loss: {test_loss}, Test Precision: {precision}, Test Recall: {recall}, Test F1: {f1}, Test AuC: {auc}")
        
# Validation function
def validate(model, val_loader, config):
    model.eval()
    val_loss = 0.0
    all_predictions, all_targets = [], []

    with torch.no_grad():
        for data, targets, _ in val_loader:
            data, targets = data.to(config['device']), targets.to(config['device'])
            outputs = model(data)
            pos_weight = targets * config['positive_weight_factor']
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            val_loss += criterion(outputs, targets).item()
            
            predictions = torch.sigmoid(outputs).cpu().numpy()
            top_k_indices = np.argsort(-predictions, axis=1)[:, :25]
            thresholded_predictions = np.zeros_like(predictions)
            rows = np.arange(len(predictions))[:, None]
            thresholded_predictions[rows, top_k_indices] = 1

            all_predictions.append(thresholded_predictions)
            all_targets.append(targets.cpu().numpy())
    
    val_loss /= len(val_loader)
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_predictions, average='samples', zero_division="warn")
    print(f"Validation Loss: {val_loss}, F1: {f1}, P: {precision}, R: {recall}")
    
    return val_loss, f1, precision, recall

# Function to split the data into training and validation sets
def split_train_val(train_metadata, val_size=0.05, seed=42):
    train_subset, val_subset = train_test_split(train_metadata, test_size=val_size, random_state=seed)
    return train_subset, val_subset

def run_experiment(train_metadata, val_metadata, test_metadata, train_transform, test_transform, param_combinations):
    
    for seed in [1, 66, 123, 777, 999]:  # Random seed for different experiment runs
        
        # Iterate over hyperparameter combinations
        for params in param_combinations:
            config = {
                'architecture': 'ResNet6',
                'predictor': 'BioclimCubes',
                'batch_size': params['batch_size'],
                'lr': params['lr'],
                'num_epochs': params['num_epochs'],
                'positive_weight_factor': params['positive_weight_factor'],
                'optimizer': params['optimizer'],  # Add optimizer to the config
                'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
            }

            # Initialize Weights and Biases (W&B) for logging
            initialize_wandb(config, run_name=f"{config['architecture']}-{config['predictor']}-seed_{seed}_opt_{params['optimizer']}_lr_{params['lr']}_batch_{params['batch_size']}_weight{params['positive_weight_factor']}")
            
                        
            # Create the train and validation datasets
            train_dataset = CustomDataset(data_path_PA, train_metadata, subset="train", transform=transform)
            val_dataset = CustomDataset(data_path_PA, val_metadata, subset="val", transform=transform)

            # Create DataLoader instances for train and validation
            train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=8)
            val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=8)

            # Create the test dataset and loader only once (as it's fixed)
            test_dataset = CustomDataset(test_data_path, test_metadata, subset="test", transform=test_transform)
            test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=8)

            # Initialize the model, optimizer, scheduler, and early stopping
            model = ResNet6(NUM_CLASSES).to(config['device'])
            early_stopping = EarlyStopping(patience=5, verbose=True, path=f"./models/{config['architecture']}-{config['predictor']}-bestF1-seed_{seed}_opt_{params['optimizer']}_lr_{params['lr']}_batch_{params['batch_size']}_weight{params['positive_weight_factor']}.pth")

            # Train and validate
            set_seed(seed)
            train_and_validate(model, train_loader, val_loader, config, early_stopping)

            test_model(model, test_loader, config, seed)
                
            print(f"Finished training for seed {seed}.")
            
            
# Initialize Weights and Biases
def initialize_wandb(config, run_name="experiment", project_name="GeoPlant"):
    wandb.init(project=project_name, name=run_name, config=config)

In [8]:
class CustomDataset(Dataset):
    def __init__(self, data_dir, metadata, subset, transform=None):
        self.subset = subset
        self.transform = transform
        self.data_dir = data_dir
        self.metadata = metadata
        self.metadata = self.metadata.dropna(subset="speciesId").reset_index(drop=True)
        self.metadata['speciesId'] = self.metadata['speciesId'].astype(int)
        self.label_dict = self.metadata.groupby('surveyId')['speciesId'].apply(list).to_dict()
        
        self.metadata = self.metadata.drop_duplicates(subset="surveyId").reset_index(drop=True)


    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        
        survey_id = self.metadata.surveyId[idx]
        if self.subset in ["train", "val"]:
            sample = torch.nan_to_num(torch.load(os.path.join(self.data_dir, f"{survey_id}_cube.pt")))
        else:
            sample = torch.nan_to_num(torch.load(os.path.join(self.data_dir, f"{survey_id}_cube.pt")))

        species_ids = self.label_dict.get(survey_id, [])  # Get list of species IDs for the survey ID
        label = torch.zeros(NUM_CLASSES)  # Initialize label tensor
        for species_id in species_ids:
            #label_id = self.species_mapping[species_id]  # Get consecutive integer label
            label_id = species_id
            label[label_id] = 1  # Set the corresponding class index to 1 for each species

        # Ensure the sample is in the correct format for the transform
        if isinstance(sample, torch.Tensor):
            sample = sample.permute(1, 2, 0)  # Change tensor shape from (C, H, W) to (H, W, C)
            sample = sample.numpy()  # Convert tensor to numpy array
            #print(sample.shape)

        if self.transform:
            sample = self.transform(sample)

        return sample, label, survey_id

In [9]:
# Initialize Weights and Biases
def initialize_wandb(config, run_name="experiment", project_name="GeoPlant"):
    wandb.init(project=project_name, name=run_name, config=config)

In [0]:
import torch.optim as optim
import itertools

NUM_CLASSES = 11255 # Number of unique classes

transform = transforms.Compose([
    transforms.ToTensor()
])

# File paths for training and testing data
data_path_PA = "../../dataset/Bioclimatic/cubes-v2/PA-train-bioclimatic_monthly/"
label_path_PA = "../../metadata/GLC24_PA_metadata_train.csv"



dev_metadata = pd.read_csv(label_path_PA)

val_surveys = dev_metadata.drop_duplicates(subset="surveyId").sample(frac=0.05, random_state=1).surveyId.unique()

val_metadata = dev_metadata[dev_metadata.surveyId.isin(val_surveys)].reset_index(drop=True)
train_metadata = dev_metadata[~dev_metadata.surveyId.isin(val_surveys)].reset_index(drop=True)

test_data_path = "../../dataset/Bioclimatic/cubes-v2/PA-test-bioclimatic_monthly/"
test_metadata_path = "../../metadata/PA_metadata_test.csv"
test_labels_path = "../../metadata/test_labels.csv"
test_metadata = pd.read_csv(test_metadata_path)
test_labels = pd.read_csv(test_labels_path)
test_labels["speciesId"] = test_labels["predictions"].str.split()
test_metadata = (
    test_metadata.merge(test_labels[["surveyId", "speciesId"]], on="surveyId", how="inner")
    .explode("speciesId")
    .dropna(subset=["speciesId"])
    .reset_index(drop=True)
)
test_metadata["speciesId"] = test_metadata["speciesId"].astype(int)


param_grid = {
    'batch_size': [64],
    'lr': [0.0001],
    'num_epochs': [100],
    'positive_weight_factor': [1, 10],
    'optimizer': ['AdamW']  # List of optimizers to use
}

# Function to generate all combinations of hyperparameters
def create_param_combinations(param_grid):
    keys, values = zip(*param_grid.items())
    return [dict(zip(keys, combination)) for combination in itertools.product(*values)]

# Generate all possible hyperparameter combinations
param_combinations = create_param_combinations(param_grid)

# Run the experimentnum_classes
run_experiment(train_metadata, val_metadata, test_metadata, train_transform=transform, test_transform=transform, param_combinations=param_combinations)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: picekl. Use `wandb login --relogin` to force relogin


Epoch 1/100, Train Loss: 0.019994601765245673
Validation Loss: 0.006272592368934835, F1: 0.20344176832302993, P: 0.1759856147448865, R: 0.28407340053317137
Validation loss decreased (inf --> 0.006273).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.006272592368934835, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [0.0001]}
Epoch 2/100, Train Loss: 0.005873884717714575
Validation Loss: 0.005601935760517205, F1: 0.2445084766153444, P: 0.2108338952573612, R: 0.34523678150603393
Validation loss decreased (0.006273 --> 0.005602).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.005601935760517205, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08,

test/AUC,▁
test/loss,▁
test/sF1@25,▁
test/sPrecision@25,▁
test/sRecall@25,▁
train/loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,████████████▆▆▆▆▆▆▆▄▄▄▄▂▂▁
val/loss,█▅▄▃▃▃▂▂▂▂▃▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁
val/sF1@25,▁▃▅▅▅▆▇▇▇▇▅▇▇▇▇▇▇▇▇███████
val/sPrecision@25,▁▃▅▅▅▆▇▇▇▇▅▇▇▇▇▆█▇▇███████
val/sRecall@25,▁▃▄▅▅▆▇▇▇▇▅▇▆▇▇▇▇▇▇█████▇█


Epoch 1/100, Train Loss: 0.043024656782314506
Validation Loss: 0.02678821509970086, F1: 0.24570649555756097, P: 0.21296471117104968, R: 0.3434812206845408
Validation loss decreased (inf --> 0.026788).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.02678821509970086, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [0.0001]}
Epoch 2/100, Train Loss: 0.024950100591210233
Validation Loss: 0.024105966676558767, F1: 0.27725648271353637, P: 0.23771634075073053, R: 0.4013205630582487
Validation loss decreased (0.026788 --> 0.024106).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.024105966676558767, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 

test/AUC,▁
test/loss,▁
test/sF1@25,▁
test/sPrecision@25,▁
test/sRecall@25,▁
train/loss,█▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,████████████▇▇▇▇▅▅▅▄▄▄▄▃▃▃▂▂▁
val/loss,█▅▄▄▅▃▃▂▃▂▃▃▂▂▂▃▁▂▄▃▁▂▂▁▁▂▂▁▂
val/sF1@25,▁▄▅▅▄▅▆▆▆▇▆▅▇▇▇▅▇▆▆▆█▇▇██▇▇██
val/sPrecision@25,▁▄▄▅▃▅▆▆▆▇▅▅▇▇▇▅▇▆▅▆█▇▆██▇▇██
val/sRecall@25,▁▄▅▅▄▆▆▆▇▆▆▅▇▇▇▆▇▇▆▇█▇▇██▇▇██


...

Epoch 1/100, Train Loss: 0.043137900015222226
Validation Loss: 0.026596592898879733, F1: 0.24913042548988534, P: 0.2136390200044954, R: 0.35497525624051524
Validation loss decreased (inf --> 0.026597).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.026596592898879733, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [0.0001]}
Epoch 2/100, Train Loss: 0.024984740391994047
Validation Loss: 0.02425087589238371, F1: 0.2738089206535249, P: 0.23504607777028547, R: 0.39619306986432884
Validation loss decreased (0.026597 --> 0.024251).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.02425087589238371, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 

test/AUC,▁
test/loss,▁
test/sF1@25,▁
test/sPrecision@25,▁
test/sRecall@25,▁
train/loss,█▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,███████████▆▆▅▅▅▅▃▃▃▃▂▂▁
val/loss,█▅▄▃▃▄▃▂▂▃▃▄▄▂▁▂▃▂▁▂▂▁▂▁
val/sF1@25,▁▃▄▅▆▅▅▆▇▅▅▅▃▇▇▆▅▇█▇██▇█
val/sPrecision@25,▁▃▄▅▆▅▆▆▇▅▅▄▄▇▇▇▅▇█▇██▇█
val/sRecall@25,▁▃▄▅▆▅▆▆▆▅▅▆▃▇▇▆▅▇█▇██▇█


Epoch 1/100, Train Loss: 0.01994857942631904
Validation Loss: 0.006273584114387632, F1: 0.20371730443808142, P: 0.17692065632726456, R: 0.28243102977109
Validation loss decreased (inf --> 0.006274).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.006273584114387632, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [0.0001]}
Epoch 2/100, Train Loss: 0.005869460111324599
Validation Loss: 0.005629120348021388, F1: 0.24217881148975678, P: 0.2095302315126995, R: 0.3396202245691007
Validation loss decreased (0.006274 --> 0.005629).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.005629120348021388, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'l

test/AUC,▁
test/loss,▁
test/sF1@25,▁
test/sPrecision@25,▁
test/sRecall@25,▁
train/loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,████████████████████▅▅▅▅▃▃▁
val/loss,█▅▄▃▃▃▃▃▃▂▂▂▂▂▁▂▁▁▂▂▂▁▂▁▁▂▁
val/sF1@25,▁▃▅▅▆▅▆▆▆▆▆▆▇▇▇▆▇▇▇▇▆█▇▇█▇█
val/sPrecision@25,▁▃▅▅▆▅▆▆▆▆▆▇▇▇▇▆▇█▇▇▇█▇▇█▇█
val/sRecall@25,▁▃▄▅▆▅▆▅▆▆▆▆▇▇▇▆▇▇▇▇▆▇▇▇█▇█


Epoch 1/100, Train Loss: 0.042989830196715956
Validation Loss: 0.02665603727634464, F1: 0.24765263603046506, P: 0.21459204315576536, R: 0.34791275818681083
Validation loss decreased (inf --> 0.026656).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.02665603727634464, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [0.0001]}
Epoch 2/100, Train Loss: 0.024956077008728816
Validation Loss: 0.02526877780577966, F1: 0.25333977191480384, P: 0.21994155990110137, R: 0.36109265951825936
Validation loss decreased (0.026656 --> 0.025269).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.02526877780577966, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 

In [ ]:
wandb.finish()